# Data Read & Process

In [0]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("DataProcessing").getOrCreate()

In [0]:
df1 = spark.read.format("csv")\
    .option("header","true")\
    .option("inferSchema","true")\
    .load("/Workspace/Users/vivek223singh@gmail.com/dataEngineering/customers10mb.csv")

In [0]:
df1.show(5) 
df1.printSchema()

In [0]:
from pyspark.sql.functions import * 
df1 = df1.withColumn("registration_date", to_date(col("registration_date"),"yyyy-MM-dd"))\
    .withColumn("is_active",col("is_active").cast("boolean"))

In [0]:
df1.printSchema()

In [0]:
df1 = df1.fillna({"country":"Unknown","city":"Unknown","state":"Unknown"})

In [0]:
df1 = df1.withColumn("registration_year",year(col("registration_date")))\
    .withColumn("registration_month",month(col("registration_date")))
    

In [0]:
df1.show(6)

In [0]:
unique_city = df1.select(countDistinct("city")).collect()
unique_city[0][0]

unique_state = df1.select(countDistinct("state")).collect()
unique_state[0][0]

unique_country = df1.select(countDistinct("country")).collect()
unique_country[0][0]

In [0]:
df1.groupBy("city").count().orderBy(col("count").desc()).show()

In [0]:
df1.groupBy("city","state").count().orderBy(col("count").desc()).show()

In [0]:
#Pivot Table
df1.groupBy("city").pivot("is_active").count().show()

In [0]:
df1.show(7)

In [0]:
df1.filter(col("registration_date")>="2023-09-09").show()

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("state").orderBy(col("registration_date").desc())

df1.filter(col("registration_date")>="2023-09-09")\
    .withColumn("rank",rank().over(window_spec))\
    .select("name","state","rank","registration_date").show()

In [0]:
df1.groupBy("city").agg(
    min("registration_date").alias("oldest customer"),
    max("registration_date").alias("newest customer")
).show()


In [0]:
#Serveless doesn't support file write so use this instead
# df1.write.mode("overwrite").saveAsTable("processed_data")

#check
spark.sql("select * from processed_data").show()


#To write as parquet
#store_path="/Filestore/tables/processed_data"
# df1.write.mode("overwrite").parquet(store_path)

#JOINING AND ANALYZING CUSTOMERS AND ORDERS

In [0]:
order_path = "/Workspace/Users/vivek223singh@gmail.com/dataEngineering/orders.csv"

orders_df = spark.read\
  .format("csv")\
  .option("header","true")\
  .option("inferSchema","true")\
  .load(order_path)\

orders_df.show(5)    
orders_df.printSchema()

In [0]:
#Analysis on Orders

#Join df1 and orders_df 
customers_order = df1.join(orders_df,df1.customer_id==orders_df.customer_id,"inner").drop(orders_df.customer_id)


In [0]:
#
print("Customers:", df1.count())
print("Orders:", orders_df.count())
print("Joined:", customers_order.count())
customers_order.select(
    df1["customer_id"],
    df1["name"],
    df1["city"],
    orders_df["order_date"],
    orders_df["order_id"],
    orders_df["status"]).display()


In [0]:
#Total orders per customer

customers_order_count = customers_order.groupBy("customer_id").count().orderBy(col("count").desc())
customers_order_count.show()

In [0]:
#Total amount spend by customer

customers_order_amount = customers_order.groupBy("customer_id").agg(sum("total_amount").alias("Total_spent"))
customers_order_amount.show(5)

In [0]:
#Order status
from pyspark.sql.functions import round

total_orders = customers_order.count()
order_status =  customers_order.groupBy("status").count()\
    .withColumn("percentage",round((col("count")/total_orders)*100,2))

order_status.show()

In [0]:
# orders by month

order_month_count = customers_order.withColumn("order_month",month(col("order_date")))\
    .groupBy("order_month")\
    .count()\
    .orderBy("order_month")   

order_month_count.show()


In [0]:
#ranking customers based on amount spend
from pyspark.sql.functions import dense_rank
window_spec = Window.orderBy(col("total Spent").desc())
customers_rank=customers_order_amount.withColumn("dense_rank",dense_rank().over(window_spec))
customers_rank.show()

In [0]:
#Finding customers with high order frequency but low total spend

customers_spend_vs_order = customers_order_count.join(customers_order_amount, 'customer_id','inner')\
    .orderBy(col("count").desc(),col("Total_spent"))
customers_spend_vs_order.show(10)    

In [0]:
#Storing Output Table
# output_path = "/dbfs/mnt/your_mount_point/processed_data"
# customers_order.write.mode("overwrite").parquet(output_path)
customers_order.write.mode("overwrite").saveAsTable("Customers_order")
spark.sql("select * from customers_order").show()